# SuperMimic on Colab

One-time setup:
- Runtime, change runtime type, pick a GPU.
- Colab Secrets (key icon on the left): add `WANDB_API_KEY` and allow notebook access.

Run the cells top to bottom. Everything lives in `/content` and is rebuilt every session: the Python environment and the motion files. Checkpoints are uploaded to W&B by the training run, so nothing needs Google Drive.

## 1. GPU check

In [1]:
import subprocess
out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print(out)
assert out, "No GPU found. Change the runtime type to a GPU."
driver = out.splitlines()[0].split(",")[1].strip()
assert int(driver.split(".")[0]) >= 580, (
    f"NVIDIA driver {driver} is too old. The pinned torch build (CUDA 13) needs driver 580 or newer."
)

: 

## 2. Repos

In [3]:
# No Google Drive: motions are rebuilt each session and checkpoints go to W&B.
STORE = "/content/store"
WS = "/content/supermimic_ws"

# Sibling clones: pyproject.toml resolves mjlab from ../mjlab_research.
!mkdir -p {WS} {STORE}/data
!if [ -d {WS}/mjlab_research ]; then git -C {WS}/mjlab_research pull --ff-only; else git clone --branch main https://github.com/xSurus/mjlab_research.git {WS}/mjlab_research; fi
!if [ -d {WS}/supermimic ]; then git -C {WS}/supermimic pull --ff-only; else git clone --branch thesis https://github.com/xSurus/supermimic.git {WS}/supermimic; fi
# data/ is not in .gitignore, so it stays outside the repo and cells use absolute paths.

Already up to date.
Already up to date.


## 3. Environment

In [4]:
!pip install -q uv
!cd {WS}/supermimic && env -u PYTHONPATH uv sync --locked --group dev
out = !cd {WS}/supermimic && env -u PYTHONPATH uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
print("\n".join(out))
assert out and out[-1].endswith("True"), "The uv environment is missing or torch cannot see the GPU. Read the uv sync output above."

2.14.0+cu130 True


## 4. Weights & Biases

In [1]:
import os, getpass
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
except Exception:
    key = None
os.environ["WANDB_API_KEY"] = key or getpass.getpass("W&B API key: ")

## 5. Tests

In [ ]:
!cd {WS}/supermimic && MUJOCO_GL=disable env -u PYTHONPATH uv run pytest -q -rs

........................................................................ [ 32%]
........................................................................ [ 65%]
........................................................................ [ 97%]
.....                                                                    [100%]
=============================== warnings summary ===============================
.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491
  /content/supermimic_ws/supermimic/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
221 passed, 1 warning in 34.94s


## 6. Motions

Downloads the LAFAN1 G1 retargeted CSVs and converts them to npz under `/content/store`. Existing files are skipped.

In [5]:
!cd {WS}/supermimic && env -u PYTHONPATH uv run wandb artifact get surus/supermimic/lafan1-g1-npz:latest --root {STORE}/data/npz/lafan1

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Downloading dataset artifact surus/supermimic/lafan1-g1-npz:latest
wandb: Downloading large artifact 'lafan1-g1-npz:latest', 119.67MB. 40 files...
wandb:   40 of 40 files downloaded.  
Done. 00:00:04.8 (25.2MB/s)
wandb: Artifact downloaded to /content/store/data/npz/lafan1


## 7. Smoke run

In [ ]:
# The first update compiles with torch.compile.
# On Colab's 2 CPU cores this can print nothing for many minutes.
# That is expected. Wait for it.
# For a quick check only, add TORCHDYNAMO_DISABLE=1 before "env" to skip compilation.
# Updates are then slower.
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-envs 256 --runner.max-iterations 300 --runner.run-name colab-smoke

## 8. Single-clip training run

One clip, 2048 robots, 5000 loop steps. This gives a first trained policy and real timings on this GPU.

The policy batch is 1024, not the default 4096. The policy update backpropagates through 32 imagined steps for every rollout in the batch, and at 4096 that needs more than the T4's 15 GB (it failed at 2048 robots with the default batch). If it still runs out of memory, set `--runner.policy.batch-size 512`.

In [ ]:
# Checkpoints upload to W&B every 1000 steps. The first update compiles and can print nothing for many minutes.
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-envs 2048 --runner.policy.batch-size 1024 --runner.max-iterations 5000 --runner.run-name walk1-2048-pb1024

## 8b. DMO single-clip run

Same settings as section 8, with rollouts replayed from the simulator (`--runner.states-from sim`). Run section 8 first on the `thesis` branch: this cell switches the clone to the `dmo` branch. Watch `Policy/replay_time_mismatch` (must stay 0, otherwise the windows are misaligned) and `Policy/replay_real_frac` (share of rollout steps that come from the simulator, the rest are imagined after an episode ended).

In [ ]:
!git -C {WS}/supermimic fetch origin dmo && git -C {WS}/supermimic checkout -B dmo origin/dmo
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-envs 2048 --runner.policy.batch-size 1024 --runner.max-iterations 5000 --runner.states-from sim --runner.run-name walk1-2048-pb1024-dmo

## 8c. Single-clip comparison sweep (noisy control)

Four arms on one clip, seeds 1 to 3, same settings as section 8:
- `base`: SuperMimic.
- `fresh`: DMO's start states, imagined rollouts.
- `fresh-noisy`: the same with the collection noise (std 0.1) added to imagined actions. This tests whether DMO's noisy logged actions explain its slowdown.
- `dmo`: DMO. It also logs `Policy/replay_action_gap`. About 0.01 means no policy drift; well above means stale windows.

The first cell pins the clone to a fixed commit, so every run in the sweep uses the same code. The second runs only the runs still missing. Comment out any that already finished in W&B (each takes about 9 min on an A100 once compiled). Already done on the older commit and still valid: `walk1-base-s1`, `walk1-fresh-s1`, `walk1-dmo-s1`, `walk1-base-s2`.

In [6]:
# Fixed commit for the whole sweep. Re-running cell 4 later prints a harmless pull error on a detached HEAD.
!git -C {WS}/supermimic fetch origin dmo && git -C {WS}/supermimic checkout --detach 2fcf2a7
!git -C {WS}/supermimic rev-parse --short HEAD && git -C {WS}/mjlab_research rev-parse --short HEAD

From https://github.com/xSurus/supermimic
 * branch            dmo        -> FETCH_HEAD
HEAD is now at 2fcf2a7 Act on the two reviews: noise control, drift metric, DR-off evaluation
2fcf2a7
dd51a9f


In [ ]:
COMMON = f"--motion-file {STORE}/data/npz/lafan1/walk1_subject1.npz --num-envs 2048 --runner.policy.batch-size 1024 --runner.max-iterations 5000"
ARMS = {
    "base": "",
    "fresh": "--runner.states-from model-fresh",
    "fresh-noisy": "--runner.states-from model-fresh --runner.policy.imagine-noise 0.1",
    "dmo": "--runner.states-from sim",
}
# Noisy control first: it is the run that answers the open question.
RUNS = [
    ("fresh-noisy", 1), ("fresh-noisy", 2), ("fresh-noisy", 3),
    ("dmo", 2), ("fresh", 2),
    ("dmo", 3), ("fresh", 3), ("base", 3),
]
for arm, seed in RUNS:
    flags = ARMS[arm]
    !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 {COMMON} {flags} --env.seed {seed} --runner.run-name walk1-{arm}-s{seed}

/content/supermimic_ws/supermimic/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[INFO] Loading environment: SuperMimic-GenCoord-G1
[INFO] Using local motion file: /content/store/data/npz/lafan1/walk1_subject1.npz
Setting seed: 1
Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.18.0
Module mujoco_warp._src.smooth 43a6a77 load on device 'cuda:0' took 8358.23 ms  (compiled)
Module _nxn_broadphase__locals__kernel_32304148 3230414 load on device 'cuda:0' took 604.43 ms  (compiled)
Module _nxn_broadphase__locals__kernel_32304148 2a885b9 load on device 'cuda:0' took 1032.05 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_1854ac15 1854ac1 load on device 'cuda:0' took 1544.14 ms  (comp

### Score the sweep

Scores the latest checkpoint of every finished `walk1-<arm>-s<seed>` run on the walk1 clip, deterministic policy, randomization off. Results go to `{STORE}/eval/<run name>.json`. If a name was used twice, the newest finished run wins.

In [ ]:
import json, os, subprocess
names = [f"walk1-{a}-s{s}" for a in ("base", "fresh", "fresh-noisy", "dmo") for s in (1, 2, 3)]
env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
repo = f"{WS}/supermimic"
lookup = (
    "import json, sys, wandb; runs = wandb.Api().runs('surus/supermimic', {'state': 'finished'}, order='+created_at');"
    "print(json.dumps({r.name: r.id for r in runs if r.name in sys.argv[1:]}))"
)
out = subprocess.run(["uv", "run", "python", "-c", lookup, *names], cwd=repo, env=env, capture_output=True, text=True)
ids = json.loads(out.stdout.strip().splitlines()[-1])
print(f"found {len(ids)} finished runs")
scores = {}
for name in names:
    if name not in ids:
        continue
    out_file = f"{STORE}/eval/{name}.json"
    if not os.path.exists(out_file):
        res = subprocess.run(
            ["uv", "run", "src/supermimic/scripts/evaluate.py", "SuperMimic-GenCoord-G1",
             "--wandb-path", f"surus/supermimic/{ids[name]}",
             "--motion-file", f"{STORE}/data/npz/lafan1/walk1_subject1.npz",
             "--output-file", out_file],
            cwd=repo, env=env, capture_output=True, text=True,
        )
        if not os.path.exists(out_file):
            print(f"{name}: evaluation failed\n{res.stdout[-2000:]}\n{res.stderr[-2000:]}")
            continue
    r = json.load(open(out_file))
    scores[name] = (r["mean_normalized_return"], r["failed_clips"])
for name, (ret, fell) in scores.items():
    print(f"{name:20s} return={ret:.4f} fell={fell}")

## 9. Full run (paper settings)

The paper's LAFAN1 run: all 40 clips, 4096 envs, policy batch 4096, one NVIDIA H100 (80 GB). It reached a normalized return of 0.82 in evaluation within 1.2 h on that GPU (section 4.1). A T4 cannot hold this. In Runtime, change runtime type, pick H100 if it is offered, else A100. If an A100 runs out of memory, halve `--runner.policy.batch-size` and say so when comparing to the paper.

The loop runs until `--runner.max-iterations` or until you stop it. Checkpoints go to W&B every 1000 steps, so stopping loses nothing saved. Watch `Perf/` for the step rate and `Train/mean_reward`, which is not the paper's normalized return.

In [ ]:
import glob
clips = sorted(glob.glob(f"{STORE}/data/npz/lafan1/*.npz"))
assert len(clips) == 40, f"expected the 40 LAFAN1 clips, found {len(clips)}. Run section 6."
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/train.py SuperMimic-GenCoord-G1 --motion-file '{STORE}/data/npz/lafan1/*.npz' --num-envs 4096 --runner.max-iterations 200000 --runner.run-name lafan40-4096

## 9b. Evaluate a checkpoint

Scores a run's checkpoint the way the paper scores LAFAN1: each of the 40 clips is played once from its first frame with the deterministic policy, and the score is the episode's reward sum divided by the most it could be (section 4). It prints one line per clip, the mean normalized return and the number of clips where the robot fell. The paper's target is 0.82.

Set `RUN` to the training run (`entity/project/run_id`). `--checkpoint` takes a model artifact version such as `v3`; the default is the latest one. Running this while a training run uses the same GPU slows that run and skews its timings.

In [ ]:
RUN = "surus/supermimic/<run id>"
!cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/evaluate.py SuperMimic-GenCoord-G1 --wandb-path {RUN} --motion-file '{STORE}/data/npz/lafan1/*.npz' --output-file {STORE}/eval/latest.json

## 10. Play

In [ ]:
# A 300-iteration smoke run only saves the step-0 checkpoint, so play a longer run.
# Replace the placeholders with the W&B entity and run id.
# !cd {WS}/supermimic && env -u PYTHONPATH uv run src/supermimic/scripts/play.py SuperMimic-GenCoord-G1 --wandb-path <entity>/supermimic/<run id>